# Final Test Evaluation — Social Science Concept Integration

**Purpose:** This notebook performs the definitive held-out test evaluation of ML models for concept harmonisation in social science.  
Each model is evaluated under **three techniques** (Clustering, Pairwise, Seeded Clustering) and subjected to **five psychometric audits**:

| # | Audit | What it measures |
|---|-------|-----------------|
| 1 | **Reliability** | Standard Error of Model (SEM) under stochastic embedding noise |
| 2 | **Discriminant Validity** | False-positive rate on lexically similar negative pairs |
| 3 | **Structural Validity** | Correlation with expert ELSST graph distances (Spearman, Pearson, Point-Biserial) |
| 4 | **DIF (Rare-Word Bias)** | Recall gap between common and rare terminology |
| 5 | **Semantic Decay** | Recall degradation as keyword overlap vanishes |

> **Reproducibility:** All random operations use `SEED = 42`. Models are loaded from local HuggingFace cache. Best hyperparameters are fixed from prior cross-validation.

## 0. Environment Setup & Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*n_jobs value 1 overridden.*")
import sys
import os
from pathlib import Path

# Add parent directory to Python path and change working directory
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Change working directory to parent so relative paths in config work correctly  
os.chdir(parent_dir)
print(f"Changed working directory to: {os.getcwd()}")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from contextlib import redirect_stdout
from io import StringIO
from IPython.display import display, Markdown

# Local project modules
from scripts import model_utils_shared as shared
from scripts import model_utils_clustering as muc
from scripts import model_utils_pairwise as mup
from scripts import model_utils_seed as mus
from scripts.config import MODELS, TYPE_TOKEN, TYPE_SENTENCE, BATCH_SIZE, SEED

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Random seed     : {SEED}")

Changed working directory to: /home/rass/Desktop/SocialScience-ConceptIntegration


/home/rass/Desktop/SocialScience-ConceptIntegration/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version : 2.9.1+cpu
CUDA available  : False
Random seed     : 42


## 1. Configuration — Best Hyperparameters (from Cross-Validation)

These parameters were determined during the training phase via grid search / cross-validation.  
They are **fixed** for this final held-out evaluation and must **not** be modified.

In [2]:
# ── Models to evaluate ────────────────────────────────────────────────────────
MODELS_TO_EVAL = [
    "all-mpnet-base-v2",
    "dwulff/mpnet-personality",
    "allenai/scibert_scivocab_uncased",
    "bert-base-uncased",
]

# ── Best HDBSCAN clustering parameters (from training grid search) ───────────
BEST_CLUSTERING_PARAMS = {
    "all-mpnet-base-v2":              {"n_components": 768, "min_cluster_size": 2, "min_samples": 2},
    "dwulff/mpnet-personality":       {"n_components": 768, "min_cluster_size": 2, "min_samples": 2},
    "allenai/scibert_scivocab_uncased": {"n_components": 768, "min_cluster_size": 2, "min_samples": 2},
    "bert-base-uncased":              {"n_components": 768, "min_cluster_size": 2, "min_samples": 2},
}

# ── Best pairwise cosine similarity thresholds ───────────────────────────────
BEST_PAIRWISE_THRESHOLDS = {
    "all-mpnet-base-v2":              0.69,
    "dwulff/mpnet-personality":       0.66,
    "allenai/scibert_scivocab_uncased": 0.89,
    "bert-base-uncased":              0.90,
}

# ── Best seeded clustering parameters ────────────────────────────────────────
BEST_SEEDED_PARAMS = {
    "all-mpnet-base-v2":              {"n_initial_seeds": 250, "threshold": 0.70},
    "dwulff/mpnet-personality":       {"n_initial_seeds": 10,  "threshold": 0.70},
    "allenai/scibert_scivocab_uncased": {"n_initial_seeds": 250, "threshold": 0.85},
    "bert-base-uncased":              {"n_initial_seeds": 250, "threshold": 0.85},
}

# ── Data paths (held-out test set) ───────────────────────────────────────────
TEST_POS_PATH = "datasets/processed_datasets/elsst/test_positive_pairs.csv"
TEST_NEG_PATH = "datasets/processed_datasets/elsst/test_negative_pairs.csv"

# ── Audit parameters ────────────────────────────────────────────────────────
NOISE_LEVELS = [0.0, 0.05, 0.1, 0.2, 0.3]
N_RELIABILITY_RUNS = 5

print("✓ Configuration loaded.")

✓ Configuration loaded.


## 2. Load Test Data & Prepare Indices

We load the held-out test set (positive + negative pairs) and build the index structures needed for all downstream techniques.

In [3]:
shared.setup_reproducibility(SEED)

# Load test data
pos_df, neg_df, full_df = shared.load_test_data(TEST_POS_PATH, TEST_NEG_PATH)

display(Markdown(f"""
| Split | Count |
|-------|------:|
| Positive pairs | {len(pos_df):,} |
| Negative pairs | {len(neg_df):,} |
| **Total**      | **{len(full_df):,}** |
"""))

# Build index arrays
terms1 = full_df["term1"].astype(str).tolist()
terms2 = full_df["term2"].astype(str).tolist()
labels = full_df["label"].astype(int).values
shortest_path = full_df.get("shortest_path", pd.Series([-1] * len(full_df))).values

unique_terms = pd.unique(full_df[["term1", "term2"]].values.ravel("K")).tolist()
term_to_idx = {t: i for i, t in enumerate(unique_terms)}

idx1 = np.array([term_to_idx[t] for t in terms1], dtype=np.int32)
idx2 = np.array([term_to_idx[t] for t in terms2], dtype=np.int32)
idx1_t = torch.tensor(idx1, dtype=torch.long)
idx2_t = torch.tensor(idx2, dtype=torch.long)

pos_len = len(pos_df)

print(f"Unique terms: {len(unique_terms):,}")
print(f"Pairs with expert shortest_path ≥ 0: {(shortest_path >= 0).sum():,}")

Loading test datasets...



| Split | Count |
|-------|------:|
| Positive pairs | 1,557 |
| Negative pairs | 530,219 |
| **Total**      | **531,776** |


Unique terms: 1,876
Pairs with expert shortest_path ≥ 0: 241,280


## 3. Precompute Audit Masks

Build boolean masks for the lexical, frequency, and difficulty subsets used by Audits 2–5.  
These are computed once and reused across all models and techniques.

In [4]:
# Letter n-gram cache (3-grams) for lexical similarity
token_cache = {t: shared.letter_ngrams(t, n=3) for t in unique_terms}

# ── Audit 2: Lexical trap mask (hard negatives with Jaccard > 0.5) ───────────
lex_mask_neg = np.array([
    shared.jaccard_sim(token_cache.get(t1, set()), token_cache.get(t2, set())) > 0.5
    for t1, t2 in zip(neg_df["term1"], neg_df["term2"])
], dtype=bool)
lex_mask = np.zeros(len(full_df), dtype=bool)
lex_mask[pos_len:] = lex_mask_neg

# ── Audit 5: Easy / Hard positive masks ──────────────────────────────────────
jacc_pos = np.array([
    shared.jaccard_sim(token_cache.get(t1, set()), token_cache.get(t2, set()))
    for t1, t2 in zip(pos_df["term1"], pos_df["term2"])
])
easy_mask = np.zeros(len(full_df), dtype=bool)
hard_mask = np.zeros(len(full_df), dtype=bool)
easy_mask[:pos_len] = jacc_pos > 0.5
hard_mask[:pos_len] = jacc_pos == 0.0

# ── Audit 4: Rare / Common word frequency masks ─────────────────────────────
try:
    from wordfreq import zipf_frequency

    term_freq = {}
    for t in unique_terms:
        toks = shared.simple_tokens(t)
        term_freq[t] = float(np.mean([zipf_frequency(tok, "en") for tok in toks])) if toks else 0.0

    pos_pair_freq = np.array([
        (term_freq.get(t1, 0.0) + term_freq.get(t2, 0.0)) / 2.0
        for t1, t2 in zip(pos_df["term1"], pos_df["term2"])
    ])
    low_thr = np.quantile(pos_pair_freq, 0.10)
    high_thr = np.quantile(pos_pair_freq, 0.90)

    rare_mask = np.zeros(len(full_df), dtype=bool)
    common_mask = np.zeros(len(full_df), dtype=bool)
    rare_mask[:pos_len] = pos_pair_freq <= low_thr
    common_mask[:pos_len] = pos_pair_freq >= high_thr
    has_wordfreq = True
except ImportError:
    rare_mask = common_mask = np.zeros(len(full_df), dtype=bool)
    has_wordfreq = False
    print("⚠ wordfreq not installed — Audit 4 will be skipped.")

# ── Summary ──────────────────────────────────────────────────────────────────
display(Markdown(f"""
| Audit Mask | N |
|-----------|--:|
| Lexical trap (neg, Jaccard > 0.5) | {lex_mask.sum():,} |
| Easy positives (Jaccard > 0.5) | {easy_mask.sum():,} |
| Hard positives (Jaccard = 0.0) | {hard_mask.sum():,} |
| Common pairs (top 10 % freq) | {common_mask.sum():,} |
| Rare pairs (bottom 10 % freq) | {rare_mask.sum():,} |
"""))


| Audit Mask | N |
|-----------|--:|
| Lexical trap (neg, Jaccard > 0.5) | 65 |
| Easy positives (Jaccard > 0.5) | 152 |
| Hard positives (Jaccard = 0.0) | 517 |
| Common pairs (top 10 % freq) | 156 |
| Rare pairs (bottom 10 % freq) | 156 |


## 4. Load Models & Build Embeddings

Each model is loaded once and its embeddings are cached in memory.  
All subsequent technique evaluations and audits reuse these cached embeddings.

In [5]:
embedding_cache = {}

for model_name in MODELS_TO_EVAL:
    model_cfg = MODELS[model_name]
    model_type = model_cfg.get("type", TYPE_SENTENCE)
    display_name = model_cfg.get("display_name", model_name)

    print(f"\n{'─' * 60}")
    print(f"  {display_name}  ({model_name})")
    print(f"{'─' * 60}")

    model = shared.load_model(model_name=model_name, model_type=model_type)
    if model is None:
        print(f"  ✗ FAILED — skipping")
        continue

    embedding_cache[model_name] = shared.build_embeddings(model, unique_terms, batch_size=BATCH_SIZE)
    print(f"  ✓ Embeddings cached — shape {embedding_cache[model_name]['norm_np'].shape}")

    # Free model from GPU memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'═' * 60}")
print(f"  Models loaded: {len(embedding_cache)} / {len(MODELS_TO_EVAL)}")
print(f"{'═' * 60}")


────────────────────────────────────────────────────────────
  All-MPNet-Base-v2  (all-mpnet-base-v2)
────────────────────────────────────────────────────────────
Loading Model (all-mpnet-base-v2)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:08<00:00, 13.99it/s]


  ✓ Embeddings cached — shape (1876, 768)

────────────────────────────────────────────────────────────
  MPNet-Personality  (dwulff/mpnet-personality)
────────────────────────────────────────────────────────────
Loading Model (dwulff/mpnet-personality)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:08<00:00, 13.89it/s]


  ✓ Embeddings cached — shape (1876, 768)

────────────────────────────────────────────────────────────
  SciBERT (SciVocab)  (allenai/scibert_scivocab_uncased)
────────────────────────────────────────────────────────────
Loading Model (allenai/scibert_scivocab_uncased)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:08<00:00, 13.38it/s]


  ✓ Embeddings cached — shape (1876, 768)

────────────────────────────────────────────────────────────
  BERT Base  (bert-base-uncased)
────────────────────────────────────────────────────────────
Loading Model (bert-base-uncased)...
  Encoding 1876 unique terms...


Batches: 100%|██████████| 118/118 [00:08<00:00, 13.66it/s]


  ✓ Embeddings cached — shape (1876, 768)

════════════════════════════════════════════════════════════
  Models loaded: 4 / 4
════════════════════════════════════════════════════════════


---

## 5. Final Evaluation — Three Techniques

We evaluate every model under three harmonisation techniques using their best hyperparameters from cross-validation.  
All predictions are stored in `preds_store` for reuse in the audit sections.

### 5.1 Clustering (UMAP + HDBSCAN)

Uses `model_utils_clustering.run_hdbscan_clustering()` and `model_utils_clustering.evaluate_clustering()` with the best parameters.

In [6]:
# Storage for all predictions (used by audits later)
preds_store = {"clustering": {}, "pairwise": {}, "seeded": {}}

# ── 5.1  CLUSTERING ──────────────────────────────────────────────────────────
clustering_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    emb = embedding_cache[model_name]
    params = BEST_CLUSTERING_PARAMS[model_name]

    # Dimensionality reduction (skip if n_components >= native dim)
    n_comp = params.get("n_components")
    if n_comp is None or n_comp >= emb["norm_np"].shape[1]:
        reduced = emb["norm_np"]
    else:
        reduced, _ = muc.reduce_embeddings_with_umap(
            emb["norm_np"], n_components=n_comp, random_state=SEED
        )

    # Cluster
    cluster_labels = muc.run_hdbscan_clustering(
        reduced,
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
    )

    # Predict from cluster labels
    l1 = cluster_labels[idx1]
    l2 = cluster_labels[idx2]
    preds = ((l1 != -1) & (l1 == l2)).astype(int)

    p, r, f1, tp, fp, tn, fn = shared.compute_metrics(labels, preds)
    clustering_rows.append({
        "Model": MODELS[model_name]["display_name"],
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    })

    preds_store["clustering"][model_name] = {
        "preds": preds, "cluster_labels": cluster_labels,
        "idx1": idx1, "idx2": idx2,
    }

display(Markdown("### Results — Clustering"))
display(pd.DataFrame(clustering_rows).set_index("Model"))

### Results — Clustering

,Precision,Recall,F1,TP,FP,TN,FN
Model,,,,,,,
All-MPNet-Base-v2,0.7016,0.5029,0.5859,783,333,529886,774
MPNet-Personality,0.6845,0.4920,0.5725,766,353,529866,791
SciBERT (SciVocab),0.6099,0.2922,0.3951,455,291,529928,1102
BERT Base,0.4820,0.2832,0.3568,441,474,529745,1116


### 5.2 Pairwise (Cosine Similarity Thresholding)

Uses `model_utils_pairwise.evaluate_pairwise()` at the best threshold for each model.

In [7]:
# ── 5.2  PAIRWISE ────────────────────────────────────────────────────────────
pairwise_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    norm_t = embedding_cache[model_name]["norm_t"]
    threshold = BEST_PAIRWISE_THRESHOLDS[model_name]

    # Cosine similarity via dot product of L2-normalised vectors
    sims = (norm_t[idx1_t] * norm_t[idx2_t]).sum(dim=1).cpu().numpy()
    preds = (sims > threshold).astype(np.int8)

    p, r, f1, tp, fp, tn, fn = shared.compute_metrics(labels, preds)
    pairwise_rows.append({
        "Model": MODELS[model_name]["display_name"],
        "Threshold": f"{threshold:.2f}",
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    })

    preds_store["pairwise"][model_name] = {"preds": preds, "similarities": sims}

display(Markdown("### Results — Pairwise"))
display(pd.DataFrame(pairwise_rows).set_index("Model"))

### Results — Pairwise

,Threshold,Precision,Recall,F1,TP,FP,TN,FN
Model,,,,,,,,
All-MPNet-Base-v2,0.69,0.7365,0.4811,0.5820,749,268,529951,808
MPNet-Personality,0.66,0.7070,0.4464,0.5472,695,288,529931,862
SciBERT (SciVocab),0.89,0.4797,0.2575,0.3351,401,435,529784,1156
BERT Base,0.90,0.3911,0.1522,0.2191,237,369,529850,1320


### 5.3 Seeded Clustering

Uses `model_utils_seed.sample_unique_terms()` and `model_utils_seed.seed_clustering()` with the best number of seeds and threshold.

In [8]:
# ── 5.3  SEEDED CLUSTERING ────────────────────────────────────────────────────
seeded_rows = []

for model_name in MODELS_TO_EVAL:
    if model_name not in embedding_cache:
        continue

    norm_t = embedding_cache[model_name]["norm_t"].cpu().detach()
    term_to_embedding = {t: e for t, e in zip(unique_terms, norm_t)}
    params = BEST_SEEDED_PARAMS[model_name]

    # Sample initial seeds
    initial_seeds, remaining_terms = mus.sample_unique_terms(
        params["n_initial_seeds"], full_df, np.array(unique_terms), random_state=SEED
    )

    # Run seeded clustering (suppress verbose output)
    with redirect_stdout(StringIO()):
        seeds_cluster = mus.seed_clustering(
            initial_seeds, remaining_terms, term_to_embedding,
            threshold=params["threshold"],
        )

    # Build term → cluster mapping and predict
    term_to_cluster = {}
    for cid, (seed, members) in enumerate(seeds_cluster.items()):
        for term in members:
            term_to_cluster[term] = cid

    c1 = np.array([term_to_cluster.get(t, -1) for t in terms1])
    c2 = np.array([term_to_cluster.get(t, -1) for t in terms2])
    preds = ((c1 == c2) & (c1 != -1)).astype(int)

    p, r, f1, tp, fp, tn, fn = shared.compute_metrics(labels, preds)
    seeded_rows.append({
        "Model": MODELS[model_name]["display_name"],
        "Seeds": params["n_initial_seeds"], "θ": f"{params['threshold']:.2f}",
        "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "F1": f"{f1:.4f}",
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    })

    preds_store["seeded"][model_name] = {"preds": preds, "term_to_cluster": term_to_cluster}

display(Markdown("### Results — Seeded Clustering"))
display(pd.DataFrame(seeded_rows).set_index("Model"))

### Results — Seeded Clustering

,Seeds,θ,Precision,Recall,F1,TP,FP,TN,FN
Model,,,,,,,,,
All-MPNet-Base-v2,250,0.70,0.7587,0.4624,0.5746,720,229,529990,837
MPNet-Personality,10,0.70,0.7341,0.3937,0.5125,613,222,529997,944
SciBERT (SciVocab),250,0.85,0.1360,0.3648,0.1982,568,3607,526612,989
BERT Base,250,0.85,0.1558,0.3031,0.2058,472,2558,527661,1085


---

## 6. Audit 1 — Reliability (Stability Test)

**Objective:** Measure the Standard Error of Model (SEM) under stochastic embedding noise.

**Method:**
1. For each noise level $\sigma_\epsilon \in \{0, 0.05, 0.1, 0.2, 0.3\}$, inject Gaussian noise: $\tilde{e} = e + \sigma_\epsilon \cdot \hat\sigma \cdot z$, where $\hat\sigma$ is the per-dimension std and $z \sim \mathcal{N}(0, 1)$.
2. Re-run the technique $k = 5$ times per noise level.
3. Compute $\text{ICC}(1,1)$ across runs and $\text{SEM} = \text{SD}_{F_1} \cdot \sqrt{1 - \text{ICC}}$.

Lower SEM indicates a more stable model.

In [9]:
from sklearn.metrics import f1_score as sk_f1
from sklearn.preprocessing import normalize

def _run_reliability_for_technique(technique, model_name, emb_cache,
                                   noise_levels, n_runs):
    """Run reliability audit for one model × one technique."""
    emb_raw = emb_cache["raw_np"]
    sigma   = emb_cache["sigma"]

    params_c = BEST_CLUSTERING_PARAMS.get(model_name, {})
    thr_pw   = BEST_PAIRWISE_THRESHOLDS.get(model_name, 0.5)
    params_s = BEST_SEEDED_PARAMS.get(model_name, {})

    results = {}
    for nl in noise_levels:
        f1_runs, all_preds = [], []
        for run in range(n_runs):
            rng = np.random.default_rng(SEED + run + int(nl * 10000))
            noisy = emb_raw + (rng.normal(0, 1, emb_raw.shape) * (nl * sigma)) if nl > 0 else emb_raw
            noisy_norm_np = normalize(noisy, norm="l2", axis=1)
            noisy_norm_np = np.ascontiguousarray(noisy_norm_np, dtype=np.float64)
            noisy_norm_t  = F.normalize(torch.tensor(noisy, dtype=torch.float32), p=2, dim=1)

            if technique == "clustering":
                n_comp = params_c.get("n_components")
                red = noisy_norm_np if (n_comp is None or n_comp >= noisy_norm_np.shape[1]) else \
                      muc.reduce_embeddings_with_umap(noisy_norm_np, n_comp, random_state=SEED + run)[0]
                cl = muc.run_hdbscan_clustering(red, params_c["min_cluster_size"], params_c["min_samples"])
                p_ = ((cl[idx1] != -1) & (cl[idx1] == cl[idx2])).astype(int)

            elif technique == "pairwise":
                sims = (noisy_norm_t[idx1_t] * noisy_norm_t[idx2_t]).sum(dim=1).cpu().numpy()
                p_ = (sims > thr_pw).astype(np.int8)

            else:  # seeded
                nte = noisy_norm_t.cpu().detach()
                t2e = {t: e for t, e in zip(unique_terms, nte)}
                seeds, rem = mus.sample_unique_terms(
                    params_s.get("n_initial_seeds", 250), full_df,
                    np.array(unique_terms), random_state=SEED + run)
                with redirect_stdout(StringIO()):
                    sc = mus.seed_clustering(seeds, rem, t2e, threshold=params_s.get("threshold", 0.7))
                t2c = {}
                for cid, (sd, ms) in enumerate(sc.items()):
                    for m in ms: t2c[m] = cid
                c1 = np.array([t2c.get(t, -1) for t in terms1])
                c2 = np.array([t2c.get(t, -1) for t in terms2])
                p_ = ((c1 == c2) & (c1 != -1)).astype(int)

            f1_runs.append(sk_f1(labels, p_, zero_division=0))
            all_preds.append(p_)

        f1_arr = np.array(f1_runs)
        preds_mat = np.array(all_preds)
        icc = shared.compute_icc_oneway(
            preds_mat.sum(axis=0), (preds_mat ** 2).sum(axis=0),
            preds_mat.shape[1], n_runs)
        sd = np.std(f1_arr, ddof=1)
        sem = sd * np.sqrt(max(0, 1 - icc))
        results[nl] = {"mean_f1": f1_arr.mean(), "sd_f1": sd, "icc": icc, "sem": sem}
    return results


# ── Run for all techniques ───────────────────────────────────────────────────
audit1_records = []
for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        if model_name not in embedding_cache:
            continue
        print(f"  Reliability | {technique:10s} | {model_name} ...", end="", flush=True)
        res = _run_reliability_for_technique(
            technique, model_name, embedding_cache[model_name],
            NOISE_LEVELS, N_RELIABILITY_RUNS)
        print(" ✓")
        for nl, vals in res.items():
            audit1_records.append({
                "Technique": technique, "Model": MODELS[model_name]["display_name"],
                "Noise": nl,
                "Mean F1": f"{vals['mean_f1']:.4f}",
                "SD(F1)": f"{vals['sd_f1']:.4f}",
                "ICC": f"{vals['icc']:.4f}",
                "SEM": f"{vals['sem']:.4f}",
            })

df_audit1 = pd.DataFrame(audit1_records)
display(Markdown("### Audit 1 — Reliability Results"))

for tech in ["clustering", "pairwise", "seeded"]:
    display(Markdown(f"**{tech.upper()}**"))
    sub = df_audit1[df_audit1["Technique"] == tech].drop(columns="Technique")
    # Pivot for a compact noise-level view
    pivot = sub.pivot(index="Model", columns="Noise", values="Mean F1")
    display(pivot)

  Reliability | clustering | all-mpnet-base-v2 ... ✓
  Reliability | clustering | dwulff/mpnet-personality ... ✓
  Reliability | clustering | allenai/scibert_scivocab_uncased ... ✓
  Reliability | clustering | bert-base-uncased ... ✓
  Reliability | pairwise   | all-mpnet-base-v2 ... ✓
  Reliability | pairwise   | dwulff/mpnet-personality ... ✓
  Reliability | pairwise   | allenai/scibert_scivocab_uncased ... ✓
  Reliability | pairwise   | bert-base-uncased ... ✓
  Reliability | seeded     | all-mpnet-base-v2 ... ✓
  Reliability | seeded     | dwulff/mpnet-personality ... ✓
  Reliability | seeded     | allenai/scibert_scivocab_uncased ... ✓
  Reliability | seeded     | bert-base-uncased ... ✓


### Audit 1 — Reliability Results

**CLUSTERING**

Noise,0.00,0.05,0.10,0.20,0.30
Model,,,,,
All-MPNet-Base-v2,0.5859,0.5850,0.5860,0.5863,0.5964
BERT Base,0.3568,0.3600,0.3639,0.3655,0.3766
MPNet-Personality,0.5725,0.5723,0.5719,0.5698,0.5792
SciBERT (SciVocab),0.3951,0.3947,0.3905,0.3898,0.3885


**PAIRWISE**

Noise,0.00,0.05,0.10,0.20,0.30
Model,,,,,
All-MPNet-Base-v2,0.5820,0.5832,0.5765,0.5537,0.5077
BERT Base,0.2191,0.2165,0.2093,0.1597,0.0940
MPNet-Personality,0.5472,0.5457,0.5430,0.5275,0.4841
SciBERT (SciVocab),0.3351,0.3363,0.3339,0.3154,0.2662


**SEEDED**

Noise,0.00,0.05,0.10,0.20,0.30
Model,,,,,
All-MPNet-Base-v2,0.5729,0.5724,0.5696,0.5452,0.5159
BERT Base,0.2001,0.2015,0.2042,0.1818,0.1730
MPNet-Personality,0.5122,0.5109,0.5025,0.4790,0.4216
SciBERT (SciVocab),0.1979,0.1927,0.1946,0.1809,0.1674


---

## 7. Audit 2 — Discriminant Validity (Lexical Trap)

**Objective:** Detect whether models are fooled by surface-level orthographic similarity.

We isolate **negative pairs** whose character 3-gram Jaccard similarity exceeds 0.5 (the "lexical traps") and measure the False-Positive Rate:

$$\text{FPR} = \frac{FP}{FP + TN}$$

Lower FPR means the model correctly rejects unrelated terms despite high letter overlap.

In [10]:
audit2_records = []

for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[technique].get(model_name)
        if info is None:
            continue
        preds = info["preds"]
        sub_preds = preds[lex_mask]
        sub_true  = labels[lex_mask]
        tp, fp, tn, fn = shared.confusion_counts(sub_true, sub_preds)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        audit2_records.append({
            "Technique": technique,
            "Model": MODELS[model_name]["display_name"],
            "Subset N": int(lex_mask.sum()),
            "FP": fp, "TN": tn,
            "FPR": f"{fpr:.4f}",
        })

display(Markdown("### Audit 2 — Discriminant Validity Results"))
display(pd.DataFrame(audit2_records).set_index(["Technique", "Model"]))

### Audit 2 — Discriminant Validity Results

Subset N  FP  TN     FPR
Technique  Model                                       
clustering All-MPNet-Base-v2         65  13  52  0.2000
           MPNet-Personality         65  18  47  0.2769
           SciBERT (SciVocab)        65   7  58  0.1077
           BERT Base                 65  16  49  0.2462
pairwise   All-MPNet-Base-v2         65  20  45  0.3077
           MPNet-Personality         65  19  46  0.2923
           SciBERT (SciVocab)        65  11  54  0.1692
           BERT Base                 65   7  58  0.1077
seeded     All-MPNet-Base-v2         65  12  53  0.1846
           MPNet-Personality         65  14  51  0.2154
           SciBERT (SciVocab)        65  18  47  0.2769
           BERT Base                 65  18  47  0.2769

---

## 8. Audit 3 — Structural Validity (Map Match)

**Objective:** Assess whether model-derived distances correlate with expert-curated ELSST graph distances.

We compute three correlation coefficients:

| Correlation | Description |
|------------|-------------|
| **Spearman ρ** | Rank-order correlation (robust to non-linear monotonic relationships) |
| **Pearson r** | Linear correlation between distances |
| **Point-biserial r** | Correlation between a binary model variable (same cluster = 0, diff = 1) and continuous expert distance |

For **clustering / seeded**: binary distance (0 = same cluster, 1 = different).  
For **pairwise**: binary component membership, model-graph shortest path, and cosine distance.

In [11]:
expert_mask = shortest_path >= 0
d_expert = shortest_path[expert_mask].astype(float)
print(f"Pairs with valid expert distances: {expert_mask.sum():,}\n")

audit3_records = []

# ── Clustering & Seeded: binary model distance ──────────────────────────────
for technique in ["clustering", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        info = preds_store[technique].get(model_name)
        if info is None:
            continue

        if technique == "clustering":
            cl = info["cluster_labels"]
            l1, l2 = cl[info["idx1"]], cl[info["idx2"]]
            d_model = np.where((l1 != -1) & (l1 == l2), 0, 1).astype(float)
        else:
            t2c = info["term_to_cluster"]
            c1 = np.array([t2c.get(t, -1) for t in terms1])
            c2 = np.array([t2c.get(t, -1) for t in terms2])
            d_model = np.where((c1 != -1) & (c1 == c2), 0, 1).astype(float)

        d_m = d_model[expert_mask]
        rho_s = shared.spearman_corr(d_expert, d_m)
        rho_p, p_val_p = shared.pearson_corr(d_expert, d_m)
        rpb, p_val_pb = shared.point_biserial_corr(d_m, d_expert)

        audit3_records.append({
            "Technique": technique,
            "Model": MODELS[model_name]["display_name"],
            "Metric": "binary_dist",
            "Spearman ρ": f"{rho_s:.4f}",
            "Pearson r": f"{rho_p:.4f}",
            "Point-Biserial r": f"{rpb:.4f}" if not np.isnan(rpb) else "N/A",
            "p (Pearson)": f"{p_val_p:.2e}" if not np.isnan(p_val_p) else "N/A",
            "p (PB)": f"{p_val_pb:.2e}" if not np.isnan(p_val_pb) else "N/A",
        })

# ── Pairwise: multiple distance metrics ─────────────────────────────────────
for model_name in MODELS_TO_EVAL:
    info = preds_store["pairwise"].get(model_name)
    if info is None:
        continue
    preds = info["preds"]
    sims  = info["similarities"]

    # (a) Binary same-component via union-find
    edges_mask = preds == 1
    parent, find, union_fn = shared.union_find(len(unique_terms))
    for a, b in zip(idx1[edges_mask], idx2[edges_mask]):
        union_fn(a, b)
    roots1 = np.array([find(i) for i in idx1])
    roots2 = np.array([find(i) for i in idx2])
    same_comp = (roots1 == roots2).astype(float)

    rho_s  = shared.spearman_corr(d_expert, same_comp[expert_mask])
    rho_p, pp = shared.pearson_corr(d_expert, same_comp[expert_mask])
    rpb, ppb  = shared.point_biserial_corr(same_comp[expert_mask], d_expert)

    audit3_records.append({
        "Technique": "pairwise", "Model": MODELS[model_name]["display_name"],
        "Metric": "binary_component",
        "Spearman ρ": f"{rho_s:.4f}", "Pearson r": f"{rho_p:.4f}",
        "Point-Biserial r": f"{rpb:.4f}" if not np.isnan(rpb) else "N/A",
        "p (Pearson)": f"{pp:.2e}" if not np.isnan(pp) else "N/A",
        "p (PB)": f"{ppb:.2e}" if not np.isnan(ppb) else "N/A",
    })

    # (b) Model-graph shortest path
    model_sp = shared.model_graph_shortest_paths(
        len(unique_terms), idx1[edges_mask], idx2[edges_mask], idx1, idx2)
    valid_sp = expert_mask & (model_sp >= 0)
    rho_sp = shared.spearman_corr(shortest_path[valid_sp].astype(float), model_sp[valid_sp].astype(float))
    rp_sp, pp_sp = shared.pearson_corr(shortest_path[valid_sp].astype(float), model_sp[valid_sp].astype(float))

    audit3_records.append({
        "Technique": "pairwise", "Model": MODELS[model_name]["display_name"],
        "Metric": "graph_shortest_path",
        "Spearman ρ": f"{rho_sp:.4f}", "Pearson r": f"{rp_sp:.4f}",
        "Point-Biserial r": "N/A (continuous)",
        "p (Pearson)": f"{pp_sp:.2e}" if not np.isnan(pp_sp) else "N/A",
        "p (PB)": "—",
    })

    # (c) Cosine distance
    cos_dist = 1.0 - sims
    rho_cd = shared.spearman_corr(d_expert, cos_dist[expert_mask])
    rp_cd, pp_cd = shared.pearson_corr(d_expert, cos_dist[expert_mask])

    audit3_records.append({
        "Technique": "pairwise", "Model": MODELS[model_name]["display_name"],
        "Metric": "cosine_distance",
        "Spearman ρ": f"{rho_cd:.4f}", "Pearson r": f"{rp_cd:.4f}",
        "Point-Biserial r": "N/A (continuous)",
        "p (Pearson)": f"{pp_cd:.2e}" if not np.isnan(pp_cd) else "N/A",
        "p (PB)": "—",
    })

display(Markdown("### Audit 3 — Structural Validity Results"))
display(pd.DataFrame(audit3_records).set_index(["Technique", "Model", "Metric"]))

Pairs with valid expert distances: 241,280



### Audit 3 — Structural Validity Results

Spearman ρ Pearson r  \
Technique  Model              Metric                                     
clustering All-MPNet-Base-v2  binary_dist             0.1042    0.1361   
           MPNet-Personality  binary_dist             0.1009    0.1318   
           SciBERT (SciVocab) binary_dist             0.0828    0.1064   
           BERT Base          binary_dist             0.0778    0.0997   
seeded     All-MPNet-Base-v2  binary_dist             0.0998    0.1308   
           MPNet-Personality  binary_dist             0.0923    0.1208   
           SciBERT (SciVocab) binary_dist             0.0894    0.1060   
           BERT Base          binary_dist             0.0678    0.0846   
pairwise   All-MPNet-Base-v2  binary_component       -0.1143   -0.1453   
                              graph_shortest_path     0.5930    0.5304   
                              cosine_distance         0.2388    0.2991   
           MPNet-Personality  binary_component       -0.1080   -0.1386   
                              graph_shortest_path     0.4488    0.4718   
                              cosine_distance         0.1504    0.2463   
           SciBERT (SciVocab) binary_component       -0.0704   -0.0808   
                              graph_shortest_path     0.4502    0.4012   
                              cosine_distance         0.1653    0.1867   
           BERT Base          binary_component       -0.0513   -0.0616   
                              graph_shortest_path     0.4060    0.2825   
                              cosine_distance         0.1688    0.1905   

                                                   Point-Biserial r  \
Technique  Model              Metric                                  
clustering All-MPNet-Base-v2  binary_dist                    0.1361   
           MPNet-Personality  binary_dist                    0.1318   
           SciBERT (SciVocab) binary_dist                    0.1064   
           BERT Base          binary_dist                    0.0997   
seeded     All-MPNet-Base-v2  binary_dist                    0.1308   
           MPNet-Personality  binary_dist                    0.1208   
           SciBERT (SciVocab) binary_dist                    0.1060   
           BERT Base          binary_dist                    0.0846   
pairwise   All-MPNet-Base-v2  binary_component              -0.1453   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   
           MPNet-Personality  binary_component              -0.1386   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   
           SciBERT (SciVocab) binary_component              -0.0808   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   
           BERT Base          binary_component              -0.0616   
                              graph_shortest_path  N/A (continuous)   
                              cosine_distance      N/A (continuous)   

                                                  p (Pearson)     p (PB)  
Technique  Model              Metric                                      
clustering All-MPNet-Base-v2  binary_dist            0.00e+00   0.00e+00  
           MPNet-Personality  binary_dist            0.00e+00   0.00e+00  
           SciBERT (SciVocab) binary_dist            0.00e+00   0.00e+00  
           BERT Base          binary_dist            0.00e+00   0.00e+00  
seeded     All-MPNet-Base-v2  binary_dist            0.00e+00   0.00e+00  
           MPNet-Personality  binary_dist            0.00e+00   0.00e+00  
           SciBERT (SciVocab) binary_dist            0.00e+00   0.00e+00  
           BERT Base          binary_dist            0.00e+00   0.00e+00  
pairwise   All-MPNet-Base-v2  binary_component       0.00e+00   0.00e+00  
                              gra

---

## 9. Audit 4 — Differential Item Functioning (Rare-Word Bias)

**Objective:** Detect systematic bias against rare / technical terminology.

$$\Delta\text{Recall} = \text{Recall}_{\text{Common}} - \text{Recall}_{\text{Rare}}$$

- **Common pairs**: top 10 % of Zipf frequency.  
- **Rare pairs**: bottom 10 % of Zipf frequency.  
- A large positive $\Delta$Recall indicates the model underperforms on rare terms.

In [12]:
if not has_wordfreq:
    display(Markdown("**⚠ Skipped** — `wordfreq` library is not installed."))
else:
    audit4_records = []
    for technique in ["clustering", "pairwise", "seeded"]:
        for model_name in MODELS_TO_EVAL:
            preds = preds_store[technique].get(model_name, {}).get("preds")
            if preds is None:
                continue
            rec_common = shared.recall_on_subset(labels, preds, common_mask)
            rec_rare   = shared.recall_on_subset(labels, preds, rare_mask)
            gap = rec_common - rec_rare
            audit4_records.append({
                "Technique": technique,
                "Model": MODELS[model_name]["display_name"],
                "Recall (Common)": f"{rec_common:.4f}",
                "Recall (Rare)": f"{rec_rare:.4f}",
                "ΔRecall": f"{gap:+.4f}",
                "N Common": int(common_mask.sum()),
                "N Rare": int(rare_mask.sum()),
            })

    display(Markdown("### Audit 4 — DIF Results"))
    display(pd.DataFrame(audit4_records).set_index(["Technique", "Model"]))

### Audit 4 — DIF Results

Recall (Common) Recall (Rare)  ΔRecall  \
Technique  Model                                                       
clustering All-MPNet-Base-v2           0.4615        0.4103  +0.0513   
           MPNet-Personality           0.4744        0.3910  +0.0833   
           SciBERT (SciVocab)          0.2628        0.2051  +0.0577   
           BERT Base                   0.3333        0.1731  +0.1603   
pairwise   All-MPNet-Base-v2           0.5064        0.3141  +0.1923   
           MPNet-Personality           0.4487        0.2692  +0.1795   
           SciBERT (SciVocab)          0.2372        0.1410  +0.0962   
           BERT Base                   0.1410        0.0833  +0.0577   
seeded     All-MPNet-Base-v2           0.4359        0.3205  +0.1154   
           MPNet-Personality           0.3718        0.2372  +0.1346   
           SciBERT (SciVocab)          0.3526        0.2308  +0.1218   
           BERT Base                   0.2821        0.1346  +0.1474   

                               N Common  N Rare  
Technique  Model                                 
clustering All-MPNet-Base-v2        156     156  
           MPNet-Personality        156     156  
           SciBERT (SciVocab)       156     156  
           BERT Base                156     156  
pairwise   All-MPNet-Base-v2        156     156  
           MPNet-Personality        156     156  
           SciBERT (SciVocab)       156     156  
           BERT Base                156     156  
seeded     All-MPNet-Base-v2        156     156  
           MPNet-Personality        156     156  
           SciBERT (SciVocab)       156     156  
           BERT Base                156     156

---

## 10. Audit 5 — Semantic Decay (Semantic Gap Test)

**Objective:** Test robustness as keyword overlap vanishes.

$$\text{Slope} = \text{Recall}_{\text{Hard}} - \text{Recall}_{\text{Easy}}$$

- **Easy positives**: Jaccard > 0.5 (lexical anchors — the model can "cheat" with surface cues).  
- **Hard positives**: Jaccard = 0.0 (semantic gap — requires genuine semantic understanding).  
- A negative slope indicates performance degrades on purely-semantic pairs.

In [13]:
audit5_records = []

for technique in ["clustering", "pairwise", "seeded"]:
    for model_name in MODELS_TO_EVAL:
        preds = preds_store[technique].get(model_name, {}).get("preds")
        if preds is None:
            continue
        rec_easy = shared.recall_on_subset(labels, preds, easy_mask)
        rec_hard = shared.recall_on_subset(labels, preds, hard_mask)
        slope = rec_hard - rec_easy
        audit5_records.append({
            "Technique": technique,
            "Model": MODELS[model_name]["display_name"],
            "Recall (Easy)": f"{rec_easy:.4f}",
            "Recall (Hard)": f"{rec_hard:.4f}",
            "Slope": f"{slope:+.4f}",
            "N Easy": int(easy_mask.sum()),
            "N Hard": int(hard_mask.sum()),
        })

display(Markdown("### Audit 5 — Semantic Decay Results"))
display(pd.DataFrame(audit5_records).set_index(["Technique", "Model"]))

### Audit 5 — Semantic Decay Results

Recall (Easy) Recall (Hard)    Slope  N Easy  \
Technique  Model                                                             
clustering All-MPNet-Base-v2         0.8684        0.2708  -0.5976     152   
           MPNet-Personality         0.8816        0.2495  -0.6321     152   
           SciBERT (SciVocab)        0.6447        0.0600  -0.5848     152   
           BERT Base                 0.6053        0.0812  -0.5240     152   
pairwise   All-MPNet-Base-v2         0.8816        0.1683  -0.7133     152   
           MPNet-Personality         0.8816        0.1219  -0.7597     152   
           SciBERT (SciVocab)        0.5724        0.0329  -0.5395     152   
           BERT Base                 0.4079        0.0426  -0.3653     152   
seeded     All-MPNet-Base-v2         0.8289        0.1934  -0.6355     152   
           MPNet-Personality         0.8355        0.1199  -0.7156     152   
           SciBERT (SciVocab)        0.7303        0.1044  -0.6258     152   
           BERT Base                 0.6118        0.0986  -0.5132     152   

                               N Hard  
Technique  Model                       
clustering All-MPNet-Base-v2      517  
           MPNet-Personality      517  
           SciBERT (SciVocab)     517  
           BERT Base              517  
pairwise   All-MPNet-Base-v2      517  
           MPNet-Personality      517  
           SciBERT (SciVocab)     517  
           BERT Base              517  
seeded     All-MPNet-Base-v2      517  
           MPNet-Personality      517  
           SciBERT (SciVocab)     517  
           BERT Base              517

---

## 11. Consolidated Summary

A single overview table combining the core F1 metric across all models and techniques, plus a per-technique best-model highlight.

In [14]:
# ── Build consolidated F1 summary ─────────────────────────────────────────────
summary_rows = []

for tech_name, rows_list in [("Clustering", clustering_rows),
                              ("Pairwise", pairwise_rows),
                              ("Seeded", seeded_rows)]:
    for row in rows_list:
        summary_rows.append({
            "Technique": tech_name,
            "Model": row["Model"],
            "Precision": row["Precision"],
            "Recall": row["Recall"],
            "F1": row["F1"],
        })

df_summary = pd.DataFrame(summary_rows)

display(Markdown("### F1 Summary — All Models × All Techniques"))
pivot_f1 = df_summary.pivot(index="Model", columns="Technique", values="F1")
display(pivot_f1)

# ── Best model per technique ─────────────────────────────────────────────────
display(Markdown("### Best Model per Technique"))
for tech in ["Clustering", "Pairwise", "Seeded"]:
    sub = df_summary[df_summary["Technique"] == tech]
    best = sub.loc[sub["F1"].astype(float).idxmax()]
    print(f"  {tech:12s} → {best['Model']}  (F1 = {best['F1']})")

# ── Save summary to CSV ─────────────────────────────────────────────────────
import os
os.makedirs("results", exist_ok=True)
summary_path = "results/final_test_summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\n✓ Summary saved to {summary_path}")

### F1 Summary — All Models × All Techniques

Technique,Clustering,Pairwise,Seeded
Model,,,
All-MPNet-Base-v2,0.5859,0.5820,0.5746
BERT Base,0.3568,0.2191,0.2058
MPNet-Personality,0.5725,0.5472,0.5125
SciBERT (SciVocab),0.3951,0.3351,0.1982


### Best Model per Technique

  Clustering   → All-MPNet-Base-v2  (F1 = 0.5859)
  Pairwise     → All-MPNet-Base-v2  (F1 = 0.5820)
  Seeded       → All-MPNet-Base-v2  (F1 = 0.5746)

✓ Summary saved to results/final_test_summary.csv


---

## Reproducibility Notes

| Item | Value |
|------|-------|
| Random seed | `SEED = 42` across all operations |
| Test data | `datasets/processed_datasets/test_{positive,negative}_pairs.csv` |
| Models | Loaded from local HuggingFace cache (`local_files_only=True`) |
| Hyperparameters | Fixed from prior cross-validation (Section 1) |
| Correlations | Spearman ρ, Pearson r, and Point-Biserial r (Audit 3) |

**To reproduce:** Run all cells top-to-bottom in a fresh kernel.  
All utility functions live in `model_utils_shared.py`, `model_utils_clustering.py`, `model_utils_pairwise.py`, and `model_utils_seed.py`.